# Section 3 — Quantization Benchmark (Google Colab · T4 GPU)

**Electro Pi AI Engineer Technical Test**

## Project Overview
This notebook runs the Section 3 benchmark end to end on a free Google Colab
**T4 GPU**. It loads one open-weight model — `Qwen/Qwen2.5-1.5B-Instruct` — twice
on the **same hardware, tokenizer, generation settings, and five prompts**,
differing only in weight precision:

- **Variant A — FP16 / BF16** — unquantized baseline.
- **Variant B — 4-bit NF4** — bitsandbytes, with double quantization.

Every number is measured at runtime by the project's own harness
(`src/benchmark.py`). Nothing is estimated, interpolated, or hard-coded.

## Objective
Quantify the memory / latency / throughput / quality trade-off of 4-bit NF4
quantization against the full-precision baseline, and record it as a
reproducible report.

## Hardware (target)
Google Colab **T4 GPU** (~16 GB VRAM). The 4-bit path requires CUDA — bitsandbytes
NF4 kernels are GPU-only.

## Software Versions
Pinned in `requirements.txt`: `torch>=2.1`, `transformers>=4.44`,
`accelerate>=0.33`, `bitsandbytes>=0.43`, `psutil>=5.9`. Exact runtime versions
are captured automatically in the report's Environment block — never guessed.

## Expected Outputs
- `results/comparison_report.md` — environment, methodology, comparison table,
  and side-by-side outputs for all five prompts.
- `results/raw_measurements.json` — every per-prompt timing, memory figure,
  token count, and generated string.

## Benchmark Methodology

### Experiment Design
One model, one tokenizer, five fixed prompts, greedy decoding
(`do_sample=False`). The only independent variable is weight precision, so any
difference in output or speed is attributable to quantization rather than
sampling noise or input drift. The prompts probe distinct failure modes: factual
recall, arithmetic reasoning, code generation, summarization, and structured JSON.

### Measurement Procedure
- A **warm-up** generation runs before timing so kernel-autotune / CUDA-graph
  costs are not charged to whichever variant loads first.
- **TTFT** is measured with a `max_new_tokens=1` call, isolating prefill from decode.
- Each prompt is generated **`REPEATS` times (default 3)**; the **median**
  wall-clock time is reported.
- `torch.cuda.synchronize()` brackets every timed region (CUDA is asynchronous).
- Throughput is computed over the **decode phase only** (total - prefill).

### Metrics Collected
Parameter footprint (real packed bytes), peak VRAM
(`torch.cuda.max_memory_allocated`), load time, time-to-first-token, decode
tokens/sec, total generation time, and the generated text for each prompt.

> **One-command alternative:** everything below is equivalent to running
> `!python results/generate_results.py`, which benchmarks FP16 then INT4 and
> writes both artifacts in a single call. The step-by-step cells are kept for
> transparency.

## 1. GPU check
Confirm a CUDA GPU is attached (*Runtime -> Change runtime type -> **T4 GPU***).

In [ ]:
!nvidia-smi

## 2. Clone the repository
Set `REPO_URL` to your repo. No remote? Upload the `section3-quantization` folder via the Files panel, then `os.chdir` into it and skip this cell.

In [ ]:
REPO_URL = "https://github.com/<YOUR_USERNAME>/electro-pi-ai-engineer-assessment.git"  # <-- edit me
import os
repo_dir = "electro-pi-ai-engineer-assessment"
if not os.path.isdir(repo_dir):
    !git clone $REPO_URL
os.chdir(f"{repo_dir}/section3-quantization")
print("Working directory:", os.getcwd())
!ls

## 3. Install CUDA PyTorch
Colab usually ships a CUDA build; pin cu121 for reproducibility.

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

## 4. Install project requirements

In [ ]:
!pip install -q -r requirements.txt

## 5. Verify CUDA

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No CUDA GPU. Set Runtime -> Change runtime type -> T4 GPU."
print("device:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)

## 6. Download the model (warm the cache)
Caches `Qwen2.5-1.5B-Instruct` (~3 GB) once so the timed load reads from disk.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
from src.config import config
from huggingface_hub import snapshot_download
snapshot_download(config.model_id)
print("Cached:", config.model_id)

## 7. Run the FP16 / BF16 benchmark

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
from src.benchmark import benchmark_variant, describe_environment
environment = describe_environment()
fp16 = benchmark_variant("fp16")
print(f"FP16: {fp16.param_mb:.1f} MB params, {fp16.mean_tokens_per_s:.1f} tok/s mean, {fp16.median_ttft_s:.3f}s median TTFT")

## 8. Run the INT4 (bitsandbytes NF4) benchmark

In [ ]:
int4 = benchmark_variant("int4")
print(f"INT4: {int4.param_mb:.1f} MB params, {int4.mean_tokens_per_s:.1f} tok/s mean, {int4.median_ttft_s:.3f}s median TTFT")

## 9 & 10. Generate `comparison_report.md` and `raw_measurements.json`

In [ ]:
from src.report import write_reports
json_path, md_path = write_reports({"fp16": fp16, "int4": int4}, environment, config.results_dir)
print("Wrote:", md_path)
print("Wrote:", json_path)

## 11. Display the benchmark summary

In [ ]:
from IPython.display import Markdown, display
display(Markdown(md_path.read_text(encoding="utf-8")))

## 12. Download all generated files

In [ ]:
from google.colab import files
files.download(str(md_path))
files.download(str(json_path))

## 13 & 14. Execution environment and hardware

In [ ]:
import json
print(json.dumps(environment, indent=2))
!nvidia-smi

## 15. Completion

In [ ]:
print("=" * 60)
print("Section 3 benchmark complete.")
print(f"  FP16 mean throughput : {fp16.mean_tokens_per_s:6.1f} tok/s")
print(f"  INT4 mean throughput : {int4.mean_tokens_per_s:6.1f} tok/s")
print(f"  FP16 param memory    : {fp16.param_mb:8.1f} MB")
print(f"  INT4 param memory    : {int4.param_mb:8.1f} MB")
print("Artifacts: results/comparison_report.md, results/raw_measurements.json")
print("Commit those two files to finalize Section 3.")
print("=" * 60)

## Limitations
- 4-bit NF4 requires CUDA; there is no CPU path for this variant (use GGUF /
  llama.cpp for CPU quantization - see `NOTES.md`).
- Single model at a single size (1.5B). Quantization damage grows as models
  shrink, so do not extrapolate upward without re-measuring.
- Qualitative quality is judged by reading the five side-by-side outputs; no
  automated quality score is invented over such a small sample.

## Reproducibility
Deterministic greedy decoding, median-of-3 timing, fixed prompts, and a captured
environment block make the run repeatable. Re-running on the same Colab tier
reproduces the numbers within scheduler noise. Every figure derives from
`src/benchmark.py`; nothing is hand-entered.

## Conclusion
When this notebook finishes, `results/comparison_report.md` and
`results/raw_measurements.json` hold the full, measured FP16-vs-INT4 comparison.
Commit both to complete the Section 3 deliverables.